In [39]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from gdynia_srodmiescie.loading import load_short, load_long
from gdynia_srodmiescie import building_features as bf
from gdynia_srodmiescie import street_metrics as sm
from gdynia_srodmiescie import styling as st

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
short = load_short("../../data/processed/fixed/srodmiescie.csv")
long = load_long("../../data/processed/fixed/srodmiescie_long_categories.csv")
buildings = (
    short.pipe(bf.add_building_floor_flags, long)
    .pipe(bf.add_building_type)
    .pipe(bf.add_building_periods)
    .pipe(bf.add_vacancy_kind, long)
)

crosstab street x category

In [41]:
pd.crosstab(
    long.reset_index()["addr_street"],
    long.reset_index()["category"],
    normalize="columns",
).pipe(st.background_gradient, axis="index").pipe(st.percent_formatter)

category,biura,brak parteru,finanse,garaże,gastronomia,handel,kult religijny,kultura i rozrywka,mieszkaniowa,noclegi,organizacje publiczne,przemysł i infrastruktura,pustostan,transport,usługi,zdrowie
addr_street,,,,,,,,,,,,,,,,
10 Lutego,,5.3%,8.7%,,4.6%,7.2%,,,0.9%,13.3%,9.8%,,6.5%,,3.1%,4.0%
3 Maja,,,2.2%,18.2%,1.7%,2.4%,,6.1%,3.6%,,3.9%,,2.2%,,3.1%,2.0%
Aleja Jana Pawła II,3.6%,,,,4.6%,1.0%,,18.2%,,,2.0%,,,10.0%,,
Aleja Marszałka Józefa Piłsudskiego,,,,,1.1%,,,,1.8%,,2.0%,,,,,1.0%
Antoniego Abrahama,,15.8%,21.7%,18.2%,14.9%,12.0%,,15.2%,3.6%,6.7%,5.9%,,10.9%,,11.3%,7.9%
Antoniego Hryniewickiego,28.6%,,,,0.6%,,,,,,,52.4%,6.5%,,,3.0%
Arkadiusza Rybickiego,,,,,1.1%,0.3%,,,,,,9.5%,6.5%,,1.3%,2.0%
Armii Krajowej,7.1%,21.1%,,9.1%,1.7%,3.1%,42.9%,,3.6%,6.7%,9.8%,,2.2%,,0.6%,3.0%
Dworcowa,,,2.2%,,0.6%,1.0%,,,,,,,,10.0%,,


crosstab time period x type of building

In [42]:
pd.crosstab(
    buildings.reset_index()["time_period"],
    buildings.reset_index()["type"],
    normalize="columns",
).pipe(st.background_gradient, axis="index").pipe(st.percent_formatter)

type,garage,high-rise,mixed-use,other,residential,service
time_period,,,,,,
before1920,,,,,,0.8%
1920-39,0.9%,,49.1%,11.2%,16.1%,23.2%
1940-59,,24.1%,43.5%,21.2%,61.0%,23.2%
1960-89,95.3%,51.7%,,56.2%,16.1%,29.5%
1990+,3.8%,24.1%,7.5%,11.2%,6.8%,23.2%


In [43]:
columns_for_corr_1 = [
    "num_premises_wo_vacant",
    "func_entropy",
    "share_vacant",
    "share_mix_use",
    "leader_share",
]
street_metr = long.pipe(sm.street_metrics, buildings)
street_metr.loc[:, columns_for_corr_1].corr(method="spearman")

,num_premises_wo_vacant,func_entropy,share_vacant,share_mix_use,leader_share
num_premises_wo_vacant,1.000000,0.304932,0.479918,0.124520,0.053353
func_entropy,0.304932,1.000000,-0.056164,0.215579,-0.771613
share_vacant,0.479918,-0.056164,1.000000,-0.023287,0.177264
share_mix_use,0.124520,0.215579,-0.023287,1.000000,-0.247426
leader_share,0.053353,-0.771613,0.177264,-0.247426,1.000000


In [44]:
columns_for_corr_2 = [
    "num_premises_wo_vacant",
    "func_entropy",
]
street_metr.loc[lambda d: d.index != "Świętojańska", columns_for_corr_2].corr(
    method="pearson"
)

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.302331
func_entropy,0.302331,1.000000


In [45]:
street_metr.loc[:, columns_for_corr_2].corr(method="pearson")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.095844
func_entropy,0.095844,1.000000


before vs after

In [46]:
old_ids = short.loc[
    lambda e: e["date_of_checking"] < pd.to_datetime("2026-10-22")
].index

old_long = long.loc[lambda d: d.index.isin(old_ids)]
old_buildings = buildings.loc[lambda d: d.index.isin(old_ids)]

street_metr_old = old_long.pipe(sm.street_metrics, old_buildings)

func_entropy_before_after = pd.concat(
    {
        "after": street_metr.loc[:, columns_for_corr_2].assign(
            rank_after=lambda d: d["func_entropy"].rank(method="dense", ascending=False)
        ),
        "before": street_metr_old.loc[:, columns_for_corr_2].assign(
            rank_before=lambda d: d["func_entropy"].rank(
                method="dense", ascending=False
            )
        ),
    },
    axis=1,
)


func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, ("before", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.329968
func_entropy,0.329968,1.000000


In [47]:
func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, ("after", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

,num_premises_wo_vacant,func_entropy
num_premises_wo_vacant,1.000000,0.307219
func_entropy,0.307219,1.000000


In [48]:
func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, [("after", "func_entropy"), ("before", "func_entropy")]
].corr(method="spearman")

,,after,before
,,func_entropy,func_entropy
after,func_entropy,1.00000,0.98906
before,func_entropy,0.98906,1.00000


Num of all share how much of all premises are in mixed-use buildings

In [49]:
func_ground_floor = ["mixed-use", "service"]

ground_profiles = long.pipe(sm.ground_floor_profile, buildings)
ground_profiles.loc[:, func_ground_floor].pipe(
    st.background_gradient, axis="columns"
).pipe(st.percent_formatter)

type,mixed-use,service
category,,
mieszkaniowa,0.5%,
pustostan,2.9%,0.3%
usługi,19.3%,11.3%
handel,34.7%,23.0%
finanse,5.4%,3.7%
zdrowie,11.0%,8.3%
gastronomia,20.6%,16.0%
brak parteru,1.3%,1.8%
kultura i rozrywka,1.8%,7.1%


vacancy vs building condition

In [50]:
pd.crosstab(buildings["condition"], buildings["is_vacant"], normalize="index").pipe(
    st.background_gradient
).pipe(st.percent_formatter)

is_vacant,False,True
condition,,
1,62.9%,37.1%
2,96.6%,3.4%
3,94.8%,5.2%


vacancy kind vs building condition

In [51]:
pd.crosstab(buildings["condition"], buildings["vacancy_kind"], normalize="index").pipe(
    st.background_gradient
).pipe(st.percent_formatter)

vacancy_kind,full gf vacant,vacant among working,without vacant
condition,,,
1,31.4%,5.7%,62.9%
2,1.9%,1.4%,96.6%
3,2.6%,2.6%,94.8%


vacancy kind vs time period

In [52]:
pd.crosstab(
    buildings["vacancy_kind"], buildings["time_period"], normalize="columns"
).pipe(st.background_gradient).pipe(st.percent_formatter)

time_period,before1920,1920-39,1940-59,1960-89,1990+
vacancy_kind,,,,,
full gf vacant,,3.6%,3.1%,2.9%,2.6%
vacant among working,,6.1%,0.7%,,1.7%
without vacant,100.0%,90.3%,96.3%,97.1%,95.7%


streets with vacancy among working

In [53]:
pd.crosstab(
    buildings["addr_street"],
    buildings["vacancy_kind"].eq("vacant among working"),
    normalize="index",
).rename(columns={False: "no vacant among working", True: "vacant among working"}).join(
    pd.crosstab(
        buildings["addr_street"],
        buildings["vacancy_kind"].eq("vacant among working"),
        margins=True,
    ).rename(columns={"All": "sum_build_per_street"})["sum_build_per_street"]
).loc[lambda d: d["vacant among working"].gt(0)].pipe(
    st.background_gradient, subset="vacant among working"
).pipe(st.percent_formatter, subset=["no vacant among working", "vacant among working"])

,no vacant among working,vacant among working,sum_build_per_street
addr_street,,,
Antoniego Abrahama,97.1%,2.9%,70
Arkadiusza Rybickiego,77.8%,22.2%,9
Generała Józefa Bema,95.7%,4.3%,23
Skwer Tadeusza Kościuszki,91.7%,8.3%,12
Starowiejska,98.1%,1.9%,53
Świętojańska,88.7%,11.3%,106


corr type entropy x func entropy

In [54]:
types_for_corr_3 = ["type_entropy", "func_entropy", "num_premises_wo_vacant"]

building_type_shares = buildings.pipe(sm.building_type_shares)
type_entropy = building_type_shares.pipe(sm.entropy_from_shares)
street_metr_type_entropy = street_metr.join(type_entropy)
street_metr_type_entropy.loc[:, types_for_corr_3].corr(method="spearman")

,type_entropy,func_entropy,num_premises_wo_vacant
type_entropy,1.000000,0.562671,-0.004110
func_entropy,0.562671,1.000000,0.304932
num_premises_wo_vacant,-0.004110,0.304932,1.000000


In [55]:
building_type_shares.join(type_entropy["type_entropy"]).loc[
    street_metr.index
].sort_values("type_entropy", ascending=False).pipe(
    st.background_gradient, subset=building_type_shares.columns
).pipe(st.percent_formatter, subset=building_type_shares.columns).pipe(
    st.decimal_formatter, decimals=3, subset="type_entropy"
)

,garage,high-rise,mixed-use,other,residential,service,type_entropy
addr_street,,,,,,,
3 Maja,9.5%,9.5%,42.9%,,14.3%,23.8%,1.431
Jerzego Waszyngtona,4.2%,16.7%,4.2%,4.2%,33.3%,37.5%,1.430
Armii Krajowej,4.0%,4.0%,36.0%,,24.0%,32.0%,1.332
Wójta Radtkego,,15.4%,25.6%,,17.9%,41.0%,1.311
Żołnierzy I Armii Wojska Polskiego,,14.3%,42.9%,,14.3%,28.6%,1.277
Świętego Piotra,,10.0%,20.0%,,20.0%,50.0%,1.221
Stefana Batorego,,,50.0%,12.5%,12.5%,25.0%,1.213
Władysława IV,,16.2%,45.9%,5.4%,,32.4%,1.175
Kazimierza Pułaskiego,38.5%,,,,30.8%,30.8%,1.093


In [ ]:
street_metr_type_entropy.sort_values("func_entropy", ascending=False).iloc[2:].loc[
    :, ["func_entropy", "type_entropy"]
].corr(method="spearman")

,func_entropy,type_entropy
func_entropy,1.000000,0.460513
type_entropy,0.460513,1.000000


In [57]:
long.pipe(sm.street_metrics, buildings, cat_omitted=bf.NON_SERVICE).join(
    type_entropy
).loc[:, ["func_entropy", "type_entropy"]].corr(method="spearman")

,func_entropy,type_entropy
func_entropy,1.000000,0.365627
type_entropy,0.365627,1.000000
